## Data Reading

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql.types import *

In [0]:
df = spark.read.format('parquet').load('abfss://bronze@databrickse2esa.dfs.core.windows.net/orders')
df = df.withColumn('order_date', to_timestamp(col('order_date'))).withColumn('year', year(col('order_date'))).drop('_rescued_data')
display(df)

In [0]:
class WindowClass:
    def dense_rank(self, df):
        df = df.withColumn('dense_rank', dense_rank().over(Window.partitionBy('year').orderBy(col('total_amount').desc())))
        return df
    
    def rank(self, df):
        df = df.withColumn('rank', rank().over(Window.partitionBy('year').orderBy(col('total_amount').desc())))
        return df
    
    def row_number(self, df):
        df = df.withColumn('row_no', row_number().over(Window.partitionBy('year').orderBy(col('total_amount').desc())))
        return df

In [0]:
class_obj = WindowClass()
df_new = df
df_new = class_obj.dense_rank(df_new)
df_new = class_obj.rank(df_new)
df_new = class_obj.row_number(df_new)
display(df_new)

In [0]:
df.write.format('delta').mode('overwrite').save('abfss://silver@databrickse2esa.dfs.core.windows.net/orders')

In [0]:
%sql
create table if not exists databricks_cata.silver.orders_silver
using delta
location 'abfss://silver@databrickse2esa.dfs.core.windows.net/orders'

In [0]:
%sql
select * from  databricks_cata.silver.orders_silver